┌─────────────────────────────────────────────────────────┐
│         ПОЛНЫЙ PIPELINE СИСТЕМЫ                          │
└─────────────────────────────────────────────────────────┘

ЭТАП 1: Подготовка и аналитика
  ├─ 1.1 Определение критериев оценки ТЗ
  ├─ 1.2 Дизайн структуры выходных данных
  └─ 1.3 Планирование архитектуры системы

ЭТАП 2: Проектирование системы
  ├─ 2.1 Выбор модели OpenAI + API интеграция
  ├─ 2.2 Дизайн Pydantic моделей (структура)
  ├─ 2.3 Планирование обработки ошибок
  └─ 2.4 Архитектура модулей

ЭТАП 3: Разработка ядра (Core)
  ├─ 3.1 Создание Pydantic моделей
  ├─ 3.2 Реализация функции анализа с LLM
  ├─ 3.3 Интеграция OpenAI API
  └─ 3.4 Обработка structured output

ЭТАП 4: Prompting & Optimization
  ├─ 4.1 Разработка системного промпта
  ├─ 4.2 Создание контекста для анализа
  ├─ 4.3 Тестирование качества выходов
  └─ 4.4 Итерация и улучшение

ЭТАП 5: Тестирование & Валидация
  ├─ 5.1 Unit-тесты для моделей
  ├─ 5.2 Интеграционное тестирование
  ├─ 5.3 Валидация структуры выходов
  └─ 5.4 Edge cases & error handling

ЭТАП 6: Документирование & Deploy
  ├─ 6.1 Документирование API
  ├─ 6.2 Подготовка примеров использования
  ├─ 6.3 Логирование и мониторинг
  └─ 6.4 Готовность к production


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum
from datetime import datetime

# --- Enums для строгой типизации ---
class EvaluationStatus(str, Enum):
    COMPLIANT = "compliant"          # Полностью соответствует
    PARTIAL = "partial"              # Частично (есть замечания)
    NON_COMPLIANT = "non_compliant"  # Не соответствует
    MISSING = "missing"              # Отсутствует в ТЗ
    NOT_APPLICABLE = "not_applicable" # Не применимо к данному проекту

class Priority(str, Enum):
    CRITICAL = "critical" # Блокирующий фактор
    HIGH = "high"         # Высокий риск
    MEDIUM = "medium"     # Средний риск
    LOW = "low"           # Косметическое замечание

# --- Level 3: Детальные результаты ---
class CriterionResult(BaseModel):
    id: str = Field(..., description="ID или точное название критерия из Kriterii.json")
    category: str = Field(..., description="Категория (например: 'Функциональные требования')")
    score: int = Field(..., ge=0, le=10, description="Оценка от 0 до 10")
    weight: int = Field(..., ge=1, le=5, description="Важность критерия (из входного файла)")
    status: EvaluationStatus = Field(..., description="Статус проверки")
    observation: str = Field(..., description="Цитата или факт из ТЗ, обосновывающий оценку")
    gap_analysis: Optional[str] = Field(None, description="Чего именно не хватает (если не идеально)")
    recommendation: Optional[str] = Field(None, description="Что конкретно нужно дописать/исправить")

# --- Level 2: Агрегация ---
class CategorySummary(BaseModel):
    category_name: str
    compliance_percentage: float = Field(..., description="Процент соответствия (0-100%)")
    critical_issues_count: int = Field(..., description="Количество критических проблем в категории")
    summary: str = Field(..., description="Краткий вывод по категории")

class ActionItem(BaseModel):
    priority: Priority
    title: str
    description: str
    impact: str = Field(..., description="Риск, если не исправить")

# --- Level 1: Корневая структура ---
class AnalysisMetadata(BaseModel):
    analyzed_at: str = Field(..., description="ISO 8601 timestamp")
    model_version: str = Field(..., description="Версия LLM (например, gpt-4o)")
    document_hash: Optional[str] = Field(None, description="Хэш файла ТЗ для версионирования")

class TZAnalysisReport(BaseModel):
    metadata: AnalysisMetadata
    
    # Общий итог
    overall_score: float = Field(..., description="Общий балл качества ТЗ (0-100)")
    executive_summary: str = Field(..., description="Резюме для ЛПР (2-3 предложения)")
    go_no_go_verdict: bool = Field(..., description="Рекомендация: можно ли брать в работу?")

    # Приоритизированные действия (Top-5)
    top_recommendations: List[ActionItem] = Field(..., max_items=5)

    # Сводка по разделам
    category_summaries: List[CategorySummary]

    # Полный аудит (плоский список для удобства фильтрации)
    detailed_audit: List[CriterionResult]

In [ ]:
# РОЛЬ И ЗАДАЧА
Вы — Senior Technical Reviewer (Ведущий технический аналитик) с 15-летним опытом в системной архитектуре и бизнес-анализе.
Ваша задача: Провести глубокий анализ предоставленного Технического Задания (ТЗ) на предмет его качества, выявить риски и вернуть структурированный отчет.

# КОНТЕКСТ
Качественное ТЗ — фундамент успешной разработки. Нечеткие, противоречивые или неполные требования приводят к архитектурным ошибкам, срыву сроков (scope creep) и финансовым потерям. Вы — последний рубеж контроля качества перед тем, как документ попадет к разработчикам. Ваша цель — не просто критиковать, а обеспечить "инженерную чистоту" документа.

# ВХОДНЫЕ ДАННЫЕ
Анализ производится на основе следующих параметров:
1. Целевая аудитория ТЗ: {{TARGET_AUDIENCE}}
2. Строгость проверки: Высокая (Эксперт). При сомнениях трактовать в пользу наличия риска.

# КРИТЕРИИ ОЦЕНКИ
Анализируйте текст по следующим измерениям:

1. **Ясность (Clarity):**
   - Требование должно быть однозначным.
   - Отсутствие слов-паразитов ("быстро", "удобно", "качественно") без метрик.
2. **Атомарность (Atomicity):**
   - Одно требование — одна проверяемая функция.
3. **Проверяемость (Testability):**
   - Можно ли написать тест-кейс? Есть ли критерии приемки?
4. **Непротиворечивость (Consistency):**
   - Требование не должно конфликтовать с другими частями ТЗ или здравым смыслом.
5. **{{CUSTOM_CRITERIA_NAME}}**:
   - {{CUSTOM_CRITERIA_DESCRIPTION}}

# ПРОЦЕСС АНАЛИЗА
1. **Декомпозиция:** Разбейте текст на логические блоки требований.
2. **Валидация:** Проверьте каждый блок по критериям выше.
3. **Оценка риска:** Для каждого нарушения определите уровень критичности (Low, Medium, High).
4. **Формирование рекомендаций:** Предложите конкретное исправление ("Как надо").
5. **Генерация структуры:** Заполните выходной JSON согласно схеме.

# ПРАВИЛА И CONSTRAINTS
- **Строгий формат:** Ваш ответ должен быть валидным JSON объектом, соответствующим предоставленной схеме (JSON Schema). Никакого markdown форматирования или вступительных слов за пределами JSON.
- **Язык отчета:** Русский (технический стиль).
- **Фактология:** Не додумывайте требования за автора. Если контекста не хватает — помечайте это как "Блокирующий вопрос".
- **Стиль критики:** Конструктивный, безэмоциональный.
- **Null handling:** Если поле не применимо, используйте `null`, не оставляйте пустые строки.

# ПРИМЕРЫ АНАЛИЗА

**Пример 1 (Плохое требование):**
*Текст:* "Система должна быстро загружать отчеты, чтобы пользователю было удобно."
*Анализ:*
- Проблема: Нарушение критерия "Ясность" и "Проверяемость". Субъективные термины "быстро" и "удобно".
- Риск: High. Невозможно сдать работу заказчику.
- Рекомендация: "Система должна формировать PDF-отчет размером до 50 Мб за время не более 2 секунд при нагрузке 100 RPS."

**Пример 2 (Хорошее требование):**
*Текст:* "API эндпоинт POST /users должен принимать JSON-объект согласно схеме UserCreate и возвращать 201 Created с ID созданного пользователя, либо 400 Bad Request при ошибке валидации."
*Анализ:*
- Статус: OK. Требование атомарно, проверяемо и однозначно.
- Риск: None.

# ФОРМАТ ВЫВОДА
Результат должен строго соответствовать переданной JSON Schema (Function Calling / Structured Output).


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. Описание структуры (твои "Входные данные")
class Issue(BaseModel):
    severity: str = Field(..., description="Уровень риска: Low, Medium, High")
    category: str = Field(..., description="Критерий: Clarity, Atomicity, etc.")
    description: str = Field(..., description="Описание проблемы")
    suggestion: Optional[str] = Field(None, description="Как исправить (если применимо)")

class RequirementAnalysis(BaseModel):
    original_text_segment: str
    is_compliant: bool
    issues: Optional[List[Issue]] = None # Nullable, если проблем нет

class SpecReport(BaseModel):
    overall_score: int = Field(..., description="Оценка качества ТЗ от 1 до 10")
    summary: str = Field(..., description="Краткое резюме для менеджера")
    detailed_analysis: List[RequirementAnalysis]

# 2. Подготовка промпта
system_prompt_template = """
[Вставь сюда текст System Prompt, который я написал выше]
"""

# Заполняем плейсхолдеры
system_prompt = system_prompt_template.replace("{{TARGET_AUDIENCE}}", "Senior Backend Developers")
system_prompt = system_prompt.replace("{{CUSTOM_CRITERIA_NAME}}", "Безопасность")
system_prompt = system_prompt.replace("{{CUSTOM_CRITERIA_DESCRIPTION}}", "Отсутствие явных уязвимостей в логике.")

user_spec_text = "Система должна работать хорошо и поддерживать авторизацию."

# 3. Вызов API
completion = client.beta.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Проанализируй следующее ТЗ:\n\n{user_spec_text}"},
    ],
    response_format=SpecReport,
)

result = completion.choices[0].message.parsed
print(f"Score: {result.overall_score}")
print(f"Issues found: {len(result.detailed_analysis[0].issues)}")
